# Regression: ML vs Statistical Models Comparison

This notebook compares Machine Learning and Statistical models for regression problems.

## Problem Type: Regression
## Datasets:
- California Housing
- Diabetes
- Wine Quality
- Energy Consumption

## Models:
**ML Models:**
- Scikit-learn Linear Regression
- Random Forest
- Support Vector Machine
- Multi-layer Perceptron

**Statistical Models:**
- OLS Regression (statsmodels)
- Generalized Linear Model (statsmodels)


In [1]:
# Import necessary libraries
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import time
import psutil
import os

# ML Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor

# Statistical Models
import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS
from statsmodels.genmod.generalized_linear_model import GLM

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")


✅ All libraries imported successfully!


## Utility Functions

Functions for dataset loading, model evaluation, and comparison.


In [15]:
def load_dataset(dataset_name):
    """Load and preprocess datasets."""
    from sklearn.datasets import fetch_california_housing, load_diabetes, load_wine
    
    if dataset_name == 'california_housing':
        data = fetch_california_housing()
        X, y = data.data, data.target
        feature_names = data.feature_names
        
    elif dataset_name == 'diabetes':
        data = load_diabetes()
        X, y = data.data, data.target
        feature_names = data.feature_names
        
    elif dataset_name == 'wine_quality':
        # Using wine dataset for regression (predicting alcohol content)
        data = load_wine()
        X = data.data
        y = data.target  # Using target as continuous variable (can be modified)
        feature_names = data.feature_names
        
    elif dataset_name == 'energy_consumption':
        # Generate synthetic energy consumption data
        np.random.seed(42)
        n_samples = 10000
        n_features = 8
        X = np.random.randn(n_samples, n_features)
        # Create target with some relationship
        y = (X[:, 0] * 10 + X[:, 1] * 5 + X[:, 2] * 3 + 
             np.random.randn(n_samples) * 2)
        feature_names = [f'feature_{i+1}' for i in range(n_features)]
        
    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")
    
    return X, y, feature_names


def get_memory_usage():
    """Get current memory usage in MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024


def evaluate_model(model, X_train, X_test, y_train, y_test, model_name, model_type):
    """Evaluate a model and return comprehensive metrics."""
    start_time = time.time()
    memory_before = get_memory_usage()
    
    # Train model
    if model_type == 'statistical':
        # For statsmodels, add constant
        X_train_sm = sm.add_constant(X_train)
        X_test_sm = sm.add_constant(X_test)
        
        model_result = model.fit()
        
        # Predictions
        y_train_pred = model_result.predict(X_train_sm)
        y_test_pred = model_result.predict(X_test_sm)
        
        training_time = time.time() - start_time
        
        pred_start = time.time()
        y_test_pred = model_result.predict(X_test_sm)
        prediction_time = time.time() - pred_start
        
    else:  # ML model
        model.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        pred_start = time.time()
        y_test_pred = model.predict(X_test)
        prediction_time = time.time() - pred_start
        
        y_train_pred = model.predict(X_train)
    
    memory_after = get_memory_usage()
    memory_usage = memory_after - memory_before
    
    # Calculate metrics
    r2 = r2_score(y_test, y_test_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    mae = mean_absolute_error(y_test, y_test_pred)
    
    # Explainability score (0-100)
    if model_type == 'statistical':
        explainability = 100 if 'OLS' in model_name else 90
    else:
        if 'Linear' in model_name:
            explainability = 30
        elif 'Random Forest' in model_name:
            explainability = 50
        else:
            explainability = 30
    
    results = {
        'model_name': model_name,
        'model_category': 'ml' if model_type == 'ml' else 'statistical',
        'test_r2': r2,
        'test_rmse': rmse,
        'test_mae': mae,
        'training_time': training_time,
        'prediction_time': prediction_time,
        'memory_usage': memory_usage,
        'explainability_score': explainability,
        'y_test_pred': y_test_pred,
        'y_test': y_test
    }
    
    if model_type == 'statistical':
        results['model_result'] = model_result
    
    return results


print("✅ Utility functions defined!")


✅ Utility functions defined!


## Main Comparison Function


In [16]:
def run_regression_comparison(dataset_name, test_size=0.2, random_state=42):
    """Run comprehensive comparison for a regression dataset."""
    
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'='*60}\n")
    
    # Load dataset
    X, y, feature_names = load_dataset(dataset_name)
    print(f"Dataset shape: {X.shape}")
    print(f"Features: {len(feature_names)}")
    print(f"Samples: {len(y)}")
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # Scale features (important for SVM and MLP)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    all_results = []
    
    # ML Models
    ml_models = {
        'Scikit-learn Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=random_state),
        'Support Vector Machine': SVR(kernel='rbf', C=1.0, gamma='scale'),
        'Multi-layer Perceptron': MLPRegressor(hidden_layer_sizes=(100, 50), 
                                               max_iter=500, random_state=random_state)
    }
    
    print("\n📊 Training ML Models...")
    for name, model in ml_models.items():
        print(f"  - {name}...")
        if name in ['Support Vector Machine', 'Multi-layer Perceptron']:
            results = evaluate_model(model, X_train_scaled, X_test_scaled, 
                                    y_train, y_test, name, 'ml')
        else:
            results = evaluate_model(model, X_train, X_test, 
                                    y_train, y_test, name, 'ml')
        all_results.append(results)
    
    # Statistical Models
    print("\n📈 Training Statistical Models...")
    # OLS Regression
    print("  - OLS Regression...")
    X_train_sm = sm.add_constant(X_train)
    ols_model = OLS(y_train, X_train_sm)
    results = evaluate_model(ols_model, X_train, X_test, y_train, y_test, 
                            'OLS Regression (statsmodels)', 'statistical')
    all_results.append(results)
    
    # GLM
    print("  - Generalized Linear Model...")
    glm_model = GLM(y_train, X_train_sm, family=sm.families.Gaussian())
    results = evaluate_model(glm_model, X_train, X_test, y_train, y_test,
                            'Generalized Linear Model (statsmodels)', 'statistical')
    all_results.append(results)
    
    # Create results DataFrame
    results_df = pd.DataFrame([
        {
            'model_name': r['model_name'],
            'model_category': r['model_category'],
            'test_r2': r['test_r2'],
            'test_rmse': r['test_rmse'],
            'test_mae': r['test_mae'],
            'training_time': r['training_time'],
            'prediction_time': r['prediction_time'],
            'memory_usage': r['memory_usage'],
            'explainability_score': r['explainability_score']
        }
        for r in all_results
    ])
    
    return results_df, all_results, X_test, y_test


print("✅ Comparison function defined!")


✅ Comparison function defined!


In [17]:
# Define datasets
regression_datasets = ['california_housing', 'diabetes', 'wine_quality', 'energy_consumption']

# Store all results
all_dataset_results = {}

for dataset in regression_datasets:
    try:
        results_df, detailed_results, X_test, y_test = run_regression_comparison(dataset)
        all_dataset_results[dataset] = {
            'results_df': results_df,
            'detailed_results': detailed_results,
            'X_test': X_test,
            'y_test': y_test
        }
        print(f"\n✅ Completed: {dataset}")
        print(results_df[['model_name', 'model_category', 'test_r2', 'test_rmse', 'training_time']])
        print("\n" + "-"*60 + "\n")
    except Exception as e:
        print(f"❌ Error with {dataset}: {e}")
        continue

print("✅ All comparisons completed!")



Dataset: CALIFORNIA_HOUSING

Dataset shape: (20640, 8)
Features: 8
Samples: 20640

📊 Training ML Models...
  - Scikit-learn Linear Regression...
  - Random Forest...
  - Support Vector Machine...
  - Multi-layer Perceptron...

📈 Training Statistical Models...
  - OLS Regression...
  - Generalized Linear Model...

✅ Completed: california_housing
                               model_name model_category   test_r2  test_rmse  \
0          Scikit-learn Linear Regression             ml  0.575788   0.745581   
1                           Random Forest             ml  0.804850   0.505694   
2                  Support Vector Machine             ml  0.727564   0.597497   
3                  Multi-layer Perceptron             ml  0.793199   0.520571   
4            OLS Regression (statsmodels)    statistical  0.575788   0.745581   
5  Generalized Linear Model (statsmodels)    statistical  0.575788   0.745581   

   training_time  
0       0.017543  
1      10.797977  
2       5.165311  
3       